# YOLOv8 Nano 학습 기록

맥주병·맥주캔·소주병 3종류를 탐지합니다.

기존 학습 조건: YOLOv8n / 50 epochs / imgsz 640 / batch 16 / MPS / seed 0 / optimizer auto.
기존 결과를 얻은 학습 데이터: train 933장, valid 112장, test 111장.

현재 데이터셋: train 1,007장, valid 112장, test 111장(총 1,230장). 추가 74장은 train에만 반영했으며, 이 데이터로 재학습한 결과는 아직 없습니다.

기존 기록: best.pt 검증 mAP50 약 0.899, mAP50–95 약 0.732.
최종 test: Precision 0.7485, Recall 0.7212, mAP50 0.7936, mAP50–95 0.6745.
검증값은 기존 학습 출력의 반올림된 값입니다. 서로 다른 데이터 분할의 점수를 직접 비교하지 않습니다.

아래 학습·평가 출력은 이전 실행 기록입니다. 이번 셀 정리 후 다시 실행한 결과가 아닙니다.
학습과 평가 실행 스위치는 기본 False입니다. 위에서부터 실행하면 환경과 데이터를 확인하고 저장된 그림을 볼 수 있습니다.


## 1. 환경과 프로젝트 위치 확인


In [ ]:
from pathlib import Path
import sys
import torch
import ultralytics
from ultralytics import YOLO

# VS Code에서 프로젝트 폴더를 연 뒤 이 노트북을 실행합니다.
project_dir = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "train_nano.ipynb").is_file()
     and (p / "datasets/drinks_dataset").is_dir()),
    None,
)
if project_dir is None:
    raise FileNotFoundError("데이터가 있는 Final_project 폴더에서 노트북을 열어주세요.")

dataset_dir = project_dir / "datasets/drinks_dataset"
data_yaml = project_dir / "data_nano.local.yaml"
run_dir = project_dir / "runs/detect/drink_detector_n"
# 기존 Nano 학습은 MPS를 사용했습니다. 다른 컴퓨터는 사용 가능한 장치를 선택합니다.
device = "mps" if torch.backends.mps.is_available() else (0 if torch.cuda.is_available() else "cpu")

print("프로젝트:", project_dir)
print("Python:", sys.executable)
print("Ultralytics:", ultralytics.__version__, "PyTorch:", torch.__version__)
print("학습 장치:", device)


## 2. 데이터 연결 확인


In [ ]:
import yaml

# 기존 데이터셋에서 확인한 클래스 번호를 그대로 사용합니다.
data = {
    "path": str(dataset_dir.resolve()),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": 3,
    "names": {0: "beer-bottle", 1: "beer-can", 2: "soju-bottle"},
}
for split, expected in [("train", 1007), ("val", 112), ("test", 111)]:
    images_dir = dataset_dir / data[split]
    labels_dir = images_dir.parent / "labels"
    images = [p for p in images_dir.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"}]
    labels = list(labels_dir.glob("*.txt"))
    if len(images) != expected or {p.stem for p in images} != {p.stem for p in labels}:
        raise ValueError(f"{split}: 데이터 개수 또는 이미지·라벨 대응을 확인하세요.")
    print(f"{split}: {len(images)}장, 라벨 대응 확인")

# 셀 실행 시에만 개인 경로 설정을 만듭니다. 기존 다른 설정은 덮어쓰지 않습니다.
if data_yaml.exists():
    if yaml.safe_load(data_yaml.read_text(encoding="utf-8")) != data:
        raise ValueError(f"기존 설정이 다릅니다. 파일을 확인하세요: {data_yaml}")
else:
    with data_yaml.open("x", encoding="utf-8") as f:
        yaml.safe_dump(data, f, allow_unicode=True, sort_keys=False)
print("설정:", data_yaml)


## 3. 50 epoch 본 학습
기존 출력 보존. RUN_TRAINING을 True로 바꾸면 새 학습을 시작합니다.


In [10]:
# 이미 50 epoch 학습을 완료했습니다. 다시 학습할 때만 True로 바꿉니다.
RUN_TRAINING = False

if RUN_TRAINING:
    model = YOLO("yolov8n.pt")
    results = model.train(
        data=str(data_yaml),
        epochs=50,
        imgsz=640,
        batch=16,
        device=device,
        workers=0,
        seed=0,
        optimizer="auto",
        project=str(project_dir / "runs/detect"),
        name="drink_detector_n",
        exist_ok=False,
        save=True,
        plots=True,
    )
    run_dir = Path(model.trainer.save_dir)
    print("Nano 본 학습 완료:", run_dir)
else:
    print("재학습하지 않습니다. 기존 결과:", run_dir)


Ultralytics 8.4.210 🚀 Python-3.13.9 torch-2.15.0 MPS (Apple M4)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/Users/kimjaeha/Desktop/Final_project/datasets/drinks_dataset/data.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=drink_detector_n, nbs=64, nms=None

/Users/kimjaeha/Desktop/Final_project/.venv/lib/python3.13/site-packages/torch/autograd/graph.py:1113: UserWarning: scatter_reduce_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/Context.cpp:224.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
/Users/kimjaeha/Desktop/Final_project/.venv/lib/python3.13/site-packages/torch/autograd/graph.py:1113: UserWarning: index_put_with_accumulate_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered 

       1/50      4.31G     0.9931      1.901       1.19          8        640: 100% ━━━━━━━━━━━━ 59/59 1.5it/s 39.4s0.6ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 1.9it/s 2.1s0.8s
                   all        112        321      0.294       0.58      0.361      0.267

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       2/50       4.3G     0.7755     0.9356     0.9836         70        640: 0% ──────────── 0/59  1.0s

/Users/kimjaeha/Desktop/Final_project/.venv/lib/python3.13/site-packages/torch/autograd/graph.py:1113: UserWarning: scatter_reduce_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/Context.cpp:224.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
/Users/kimjaeha/Desktop/Final_project/.venv/lib/python3.13/site-packages/torch/autograd/graph.py:1113: UserWarning: index_put_with_accumulate_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered 

       2/50      4.17G     0.8907      1.077      1.103         17        640: 100% ━━━━━━━━━━━━ 59/59 1.4it/s 41.6s0.7ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 1.8it/s 2.3s0.8ss
                   all        112        321      0.602      0.453      0.508      0.328

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/50      4.32G     0.9863      1.035      1.121         75        640: 0% ──────────── 0/59  0.9s

/Users/kimjaeha/Desktop/Final_project/.venv/lib/python3.13/site-packages/torch/autograd/graph.py:1113: UserWarning: scatter_reduce_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/Context.cpp:224.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
/Users/kimjaeha/Desktop/Final_project/.venv/lib/python3.13/site-packages/torch/autograd/graph.py:1113: UserWarning: index_put_with_accumulate_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered 

... (91 outputs left out)

/Users/kimjaeha/Desktop/Final_project/.venv/lib/python3.13/site-packages/torch/autograd/graph.py:1113: UserWarning: scatter_reduce_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/Context.cpp:224.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
/Users/kimjaeha/Desktop/Final_project/.venv/lib/python3.13/site-packages/torch/autograd/graph.py:1113: UserWarning: index_put_with_accumulate_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered 

      49/50      4.14G     0.3782     0.2575     0.8829          9        640: 100% ━━━━━━━━━━━━ 59/59 1.3it/s 46.6s0.7ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 2.4it/s 1.7s0.7s
                   all        112        321      0.794      0.863      0.854      0.701

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      50/50      4.28G     0.4523     0.2608     0.9205         35        640: 0% ──────────── 0/59  0.7s

/Users/kimjaeha/Desktop/Final_project/.venv/lib/python3.13/site-packages/torch/autograd/graph.py:1113: UserWarning: scatter_reduce_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/Context.cpp:224.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
/Users/kimjaeha/Desktop/Final_project/.venv/lib/python3.13/site-packages/torch/autograd/graph.py:1113: UserWarning: index_put_with_accumulate_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered 

      50/50      4.24G     0.3677     0.2497     0.8775         14        640: 100% ━━━━━━━━━━━━ 59/59 1.4it/s 43.6s0.6ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 2.3it/s 1.7s0.7s
                   all        112        321      0.787      0.863      0.846      0.702

50 epochs completed in 15.381 hours.
Optimizer stripped from /Users/kimjaeha/Desktop/Final_project/runs/detect/drink_detector_n/weights/last.pt, 6.5MB
Optimizer stripped from /Users/kimjaeha/Desktop/Final_project/runs/detect/drink_detector_n/weights/best.pt, 6.5MB

Validating /Users/kimjaeha/Desktop/Final_project/runs/detect/drink_detector_n/weights/best.pt...
Ultralytics 8.4.210 🚀 Python-3.13.9 torch-2.15.0 MPS (Apple M4)
Model summary (fused): 73 layers, 3,133,193 parameters, 0 gradients, 8.6 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 2.2it/s 1.8s0.8s
                   all 

## 4. 모델 평가
기본 대상은 val입니다. 아래 보존된 출력은 이전 **test** 평가 결과입니다.


In [15]:
# 모델 비교에는 val을 사용합니다. test는 최종 평가할 때만 사용합니다.
RUN_EVALUATION = False
EVAL_SPLIT = "val"  # 최종 test 평가가 필요할 때 "test"로 변경

if EVAL_SPLIT not in {"val", "test"}:
    raise ValueError("EVAL_SPLIT은 val 또는 test여야 합니다.")

best_pt = run_dir / "weights/best.pt"
print("학습된 모델:", best_pt, best_pt.is_file())

if RUN_EVALUATION:
    if not best_pt.is_file():
        raise FileNotFoundError(f"모델 파일이 없습니다: {best_pt}")
    model = YOLO(str(best_pt))
    metrics = model.val(
        data=str(data_yaml),
        split=EVAL_SPLIT,
        imgsz=640,
        batch=16,
        device=device,
        workers=0,
        plots=True,
        project=str(project_dir / "runs/detect"),
        name=f"drink_detector_n_{EVAL_SPLIT}_eval",
        exist_ok=False,
    )
    print(f"Nano {EVAL_SPLIT} 결과")
    print(f"Precision: {metrics.box.mp:.4f}")
    print(f"Recall: {metrics.box.mr:.4f}")
    print(f"mAP50: {metrics.box.map50:.4f}")
    print(f"mAP50-95: {metrics.box.map:.4f}")
else:
    print("평가를 다시 실행하지 않습니다. 아래 기존 출력은 이전 test 평가 기록입니다.")


학습된 모델 존재: True
YAML 존재: True
Ultralytics 8.4.210 🚀 Python-3.13.9 torch-2.15.0 MPS (Apple M4)
Model summary (fused): 73 layers, 3,133,193 parameters, 0 gradients, 8.6 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1412.2±535.2 MB/s, size: 36.3 KB)
val: Scanning /Users/kimjaeha/Desktop/Final_project/datasets/drinks_dataset/test/labels... 111 images, 2 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 111/111 3.1Kit/s 0.0s
val: New cache created: /Users/kimjaeha/Desktop/Final_project/datasets/drinks_dataset/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 4.3it/s 1.6s0.2s
                   all        111        326      0.748      0.721      0.794      0.674
           beer-bottle         86        144      0.652      0.729      0.721      0.586
              beer-can         11         19      0.766      0.864      0.887      0.745
           soju-bottle         63        163      0.827      0.571      0

## 5. 저장된 학습 결과 그림


In [ ]:
# 저장된 결과 그림만 표시합니다. 학습이나 평가를 실행하지 않습니다.
from IPython.display import Image, display

for name in ["results.png", "confusion_matrix.png", "val_batch0_pred.jpg"]:
    path = run_dir / name
    if path.is_file():
        print(name)
        display(Image(filename=str(path), width=900))
    else:
        print("그림 없음:", path)
